# Lab 2: Manual Sharding with PostgreSQL
## ISM 6562 — Big Data for Business Applications | Week 04

---

### The Scenario

**TechMart** operates three regional warehouses — East (New Jersey), Central (Texas), and West (Oregon). Each warehouse has hundreds of IoT sensors monitoring temperature, humidity, and equipment status.

The operations team has been running all sensor data on a single PostgreSQL server. But with 150,000+ readings accumulating and three geographically distributed teams querying the data, the single server is becoming a bottleneck:

- **Disk is filling up** — one server can't hold unlimited growth
- **Query contention** — East Coast queries compete with West Coast queries
- **Latency** — the Oregon team gets slow responses from the New Jersey server

The solution: **shard the data across three PostgreSQL instances**, one per region. Each shard holds only its region's data.

In this lab, you'll build this architecture from scratch and experience the **real trade-offs** of manual sharding — both the benefits and the painful parts.

### What You'll Learn

1. How to **create identical schemas** on multiple database instances
2. **Shard routing** — directing data to the right shard
3. **Reference tables** — replicating shared data across shards
4. Why **cross-shard JOINs** and **foreign keys** don't work
5. How to implement **cross-shard aggregation** in application code
6. Using **dblink** for SQL-level cross-shard queries

### Prerequisites

Start the Docker environment:

```bash
cd sharding-lab
docker compose up -d
```

Wait for all 5 containers (coordinator + 3 shards + pgAdmin) to be healthy.

## 1. Setup: Connect to All Shards

In [ ]:
import psycopg2
import time
import random
import hashlib
from datetime import datetime, timedelta

# Shard connection parameters
SHARDS = {
    'east': {
        'host': 'localhost', 'port': 5433,
        'dbname': 'shard1_db', 'user': 'shard_user', 'password': 'shard_pass'
    },
    'central': {
        'host': 'localhost', 'port': 5434,
        'dbname': 'shard2_db', 'user': 'shard_user', 'password': 'shard_pass'
    },
    'west': {
        'host': 'localhost', 'port': 5435,
        'dbname': 'shard3_db', 'user': 'shard_user', 'password': 'shard_pass'
    }
}

COORDINATOR = {
    'host': 'localhost', 'port': 5436,
    'dbname': 'distributed_db', 'user': 'coordinator', 'password': 'coord_pass'
}

# Connect to all shards
shard_conns = {}
shard_curs = {}

for region, params in SHARDS.items():
    conn = psycopg2.connect(**params)
    conn.autocommit = True
    shard_conns[region] = conn
    shard_curs[region] = conn.cursor()
    print(f"Connected to shard: {region:8s} (port {params['port']})")

# Connect to coordinator
coord_conn = psycopg2.connect(**COORDINATOR)
coord_conn.autocommit = True
coord_cur = coord_conn.cursor()
print(f"Connected to coordinator (port {COORDINATOR['port']})")

print(f"\nAll 4 database connections established.")

## 2. Schema Creation: The Operational Burden

With sharding, there is no central schema manager. You must create **identical tables on every shard** manually. If you forget one, or make a typo, that shard has a different schema.

This is one of the operational costs of manual sharding.

In [ ]:
# Schema DDL — must be applied to EVERY shard
SCHEMA_DDL = [
    """
    CREATE TABLE IF NOT EXISTS warehouses (
        warehouse_id   INTEGER PRIMARY KEY,
        region         VARCHAR(20)  NOT NULL,
        city           VARCHAR(50)  NOT NULL,
        state          VARCHAR(2)   NOT NULL,
        capacity_sqft  INTEGER      NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS sensors (
        sensor_id      INTEGER PRIMARY KEY,
        warehouse_id   INTEGER      NOT NULL,
        sensor_type    VARCHAR(20)  NOT NULL,
        zone           VARCHAR(20)  NOT NULL,
        installed_date DATE         NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS sensor_readings (
        reading_id     SERIAL PRIMARY KEY,
        sensor_id      INTEGER        NOT NULL,
        reading_time   TIMESTAMP      NOT NULL,
        value          NUMERIC(10,2)  NOT NULL,
        unit           VARCHAR(10)    NOT NULL,
        alert_level    VARCHAR(10)    DEFAULT 'normal'
    )
    """
]

# Apply to ALL shards
for region, cur in shard_curs.items():
    for ddl in SCHEMA_DDL:
        cur.execute(ddl)
    print(f"Schema created on shard: {region}")

print(f"\nApplied {len(SCHEMA_DDL)} CREATE TABLE statements × {len(SHARDS)} shards")
print("= {} total DDL operations. Imagine doing this with 50 shards.".format(
    len(SCHEMA_DDL) * len(SHARDS)
))

### Schema Drift: What Happens When You Forget a Shard?

With manual sharding, every schema change must be applied to every shard independently. Let's see what happens when one shard falls behind.

In [ ]:
# Simulate schema drift: add a column to 2 shards but "forget" the 3rd
print("SCHEMA DRIFT DEMO: Adding a column to some shards but not all")
print("=" * 60)

# Add 'firmware_version' to east and central, but NOT west
shard_curs['east'].execute(
    "ALTER TABLE sensors ADD COLUMN IF NOT EXISTS firmware_version VARCHAR(10)"
)
shard_curs['central'].execute(
    "ALTER TABLE sensors ADD COLUMN IF NOT EXISTS firmware_version VARCHAR(10)"
)
print("Added 'firmware_version' to east and central shards.")
print("Forgot to run it on west shard!\n")

# Now try to query firmware_version on all shards
for region, cur in shard_curs.items():
    try:
        cur.execute("SELECT sensor_id, firmware_version FROM sensors LIMIT 1")
        cur.fetchone()
        print(f"  {region:8s}: firmware_version column EXISTS")
    except psycopg2.Error as e:
        print(f"  {region:8s}: ERROR — {e.pgerror.strip()}")
        shard_conns[region].rollback()

print("\nWith 3 shards this is manageable. With 50 shards across teams,")
print("schema drift becomes a serious operational risk.")

# Clean up: add the column to west too, so later cells work consistently
shard_curs['west'].execute(
    "ALTER TABLE sensors ADD COLUMN IF NOT EXISTS firmware_version VARCHAR(10)"
)
print("\n(Fixed: added column to west shard for consistency.)")

## 3. Reference Tables: Replicate Shared Data

The `warehouses` table is small and queried from every shard (e.g., to get the warehouse name in a JOIN). This is a **reference table** — we replicate it identically on all shards.

If a warehouse's city name changes, you must update it on **every shard** manually.

In [ ]:
# Warehouse reference data
WAREHOUSES = [
    (1, 'east',    'Newark',    'NJ', 250000),
    (2, 'central', 'Dallas',    'TX', 300000),
    (3, 'west',    'Portland',  'OR', 200000),
]

# Replicate to ALL shards
for region, cur in shard_curs.items():
    cur.execute("DELETE FROM warehouses")  # idempotent
    for wh in WAREHOUSES:
        cur.execute(
            "INSERT INTO warehouses VALUES (%s, %s, %s, %s, %s)",
            wh
        )
    print(f"Replicated {len(WAREHOUSES)} warehouses to shard: {region}")

print("\nEvery shard has the full warehouses table.")
print("This enables local JOINs between sensors and warehouses.")

### Reference Table Inconsistency: The Hidden Risk

Reference tables work great — until someone updates one shard but forgets another. Let's simulate what happens when the East warehouse changes its city name but the update doesn't reach all shards.

In [ ]:
# Simulate reference table inconsistency
print("REFERENCE TABLE INCONSISTENCY DEMO")
print("=" * 60)

# Update warehouse 1's city on east and central, but "forget" west
shard_curs['east'].execute(
    "UPDATE warehouses SET city = 'Elizabeth' WHERE warehouse_id = 1"
)
shard_curs['central'].execute(
    "UPDATE warehouses SET city = 'Elizabeth' WHERE warehouse_id = 1"
)
print("Updated warehouse 1 city to 'Elizabeth' on east and central.")
print("Forgot to update west shard!\n")

# Query each shard for warehouse 1's city
print("What does each shard think warehouse 1's city is?")
for region, cur in shard_curs.items():
    cur.execute("SELECT city FROM warehouses WHERE warehouse_id = 1")
    city = cur.fetchone()[0]
    status = "CORRECT" if city == "Elizabeth" else "STALE!"
    print(f"  {region:8s}: city = '{city}'  ({status})")

print("\nThe west shard has stale data. A JOIN on the west shard will")
print("show 'Newark' while the other shards show 'Elizabeth'.")
print("This is a real risk with reference tables in production.")

# Fix it: restore original city on all shards
for cur in shard_curs.values():
    cur.execute("UPDATE warehouses SET city = 'Newark' WHERE warehouse_id = 1")
print("\n(Restored all shards to 'Newark' for consistency.)")

## 4. Shard Routing: Directing Data to the Right Shard

The core question in sharding: **which shard does this row belong to?**

We'll implement **list-based routing** — each region maps to a specific shard. This is the shard routing function that your application must implement.

In [ ]:
# Shard routing function — list-based by region
REGION_TO_SHARD = {
    'east':    'east',
    'central': 'central',
    'west':    'west',
}

def get_shard_cursor(region):
    """Route to the correct shard based on region."""
    shard = REGION_TO_SHARD.get(region)
    if shard is None:
        raise ValueError(f"Unknown region: {region}")
    return shard_curs[shard]

print("List-based shard routing:")
for region, shard in REGION_TO_SHARD.items():
    print(f"  {region:10s} -> shard '{shard}' (port {SHARDS[shard]['port']})")

In [ ]:
# For comparison: hash-based routing (we'll discuss but use list-based)
def hash_route(key, num_shards=3):
    """Hash-based shard routing — distributes evenly but loses locality."""
    h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
    return h % num_shards

print("Hash-based routing example (for comparison):")
shard_names = ['east', 'central', 'west']
for sensor_id in range(1, 11):
    shard_idx = hash_route(sensor_id)
    print(f"  sensor_id={sensor_id:2d}  ->  shard '{shard_names[shard_idx]}'")

print("\nHash routing gives even distribution but sensors from the same")
print("warehouse could end up on different shards — losing data locality.")
print("We'll use list-based routing for this lab.")

## 5. Data Insertion: Route Sensors and Readings to Shards

Now we'll populate each shard with its region's sensors and generate ~50,000 readings per shard. The application is responsible for routing every INSERT to the correct shard.

In [ ]:
# Define sensors for each region
SENSOR_DEFS = {
    'east': [
        # (sensor_id, warehouse_id, sensor_type, zone, installed_date)
        (101, 1, 'temperature', 'receiving',  '2024-06-01'),
        (102, 1, 'temperature', 'storage-A',  '2024-06-01'),
        (103, 1, 'temperature', 'storage-B',  '2024-06-15'),
        (104, 1, 'humidity',    'receiving',  '2024-07-01'),
        (105, 1, 'humidity',    'storage-A',  '2024-07-01'),
        (106, 1, 'humidity',    'storage-B',  '2024-07-15'),
        (107, 1, 'temperature', 'cold-store', '2024-08-01'),
        (108, 1, 'humidity',    'cold-store', '2024-08-01'),
    ],
    'central': [
        (201, 2, 'temperature', 'dock-north', '2024-05-01'),
        (202, 2, 'temperature', 'dock-south', '2024-05-01'),
        (203, 2, 'temperature', 'bulk-store', '2024-05-15'),
        (204, 2, 'humidity',    'dock-north', '2024-06-01'),
        (205, 2, 'humidity',    'dock-south', '2024-06-01'),
        (206, 2, 'humidity',    'bulk-store', '2024-06-15'),
        (207, 2, 'temperature', 'hazmat',     '2024-07-01'),
        (208, 2, 'humidity',    'hazmat',     '2024-07-01'),
    ],
    'west': [
        (301, 3, 'temperature', 'inbound',     '2024-08-01'),
        (302, 3, 'temperature', 'outbound',    '2024-08-01'),
        (303, 3, 'temperature', 'main-floor',  '2024-08-15'),
        (304, 3, 'humidity',    'inbound',     '2024-09-01'),
        (305, 3, 'humidity',    'outbound',    '2024-09-01'),
        (306, 3, 'humidity',    'main-floor',  '2024-09-15'),
        (307, 3, 'temperature', 'mezzanine',   '2024-10-01'),
        (308, 3, 'humidity',    'mezzanine',   '2024-10-01'),
    ]
}

# Insert sensors into their respective shards
for region, sensors in SENSOR_DEFS.items():
    cur = get_shard_cursor(region)
    cur.execute("DELETE FROM sensor_readings")  # idempotent
    cur.execute("DELETE FROM sensors")
    for s in sensors:
        cur.execute(
            "INSERT INTO sensors VALUES (%s, %s, %s, %s, %s)",
            s
        )
    print(f"Inserted {len(sensors)} sensors into shard: {region}")

print(f"\nTotal: {sum(len(s) for s in SENSOR_DEFS.values())} sensors across 3 shards")

In [ ]:
# Generate sensor readings — ~50K per shard
# 8 sensors × 6250 readings each = 50,000 per shard
import math

READINGS_PER_SENSOR = 6250  # ~260 days of hourly data

random.seed(42)  # reproducible

for region, sensors in SENSOR_DEFS.items():
    cur = get_shard_cursor(region)
    count = 0
    
    for sensor_id, wh_id, stype, zone, inst_date in sensors:
        base_time = datetime(2025, 1, 1)
        
        # Build batch insert for performance
        values = []
        for i in range(READINGS_PER_SENSOR):
            ts = base_time + timedelta(hours=i)
            hour_angle = 2 * math.pi * (ts.hour / 24.0)
            
            if stype == 'temperature':
                val = round(72.0 + 5.0 * math.sin(hour_angle) + random.gauss(0, 1.5), 2)
                unit = 'F'
            else:  # humidity
                val = round(45.0 + 10.0 * math.sin(hour_angle + 0.78) + random.gauss(0, 2.0), 2)
                unit = '%'
            
            alert = 'normal'
            if stype == 'temperature' and (val > 85 or val < 55):
                alert = 'warning'
            if stype == 'humidity' and (val > 70 or val < 20):
                alert = 'warning'
            
            values.append((sensor_id, ts, val, unit, alert))
        
        # Batch insert
        args_str = ','.join(
            cur.mogrify("(%s,%s,%s,%s,%s)", v).decode() for v in values
        )
        cur.execute(
            f"INSERT INTO sensor_readings (sensor_id, reading_time, value, unit, alert_level) VALUES {args_str}"
        )
        count += READINGS_PER_SENSOR
    
    print(f"Inserted {count:,} readings into shard: {region}")

total = READINGS_PER_SENSOR * sum(len(s) for s in SENSOR_DEFS.values())
print(f"\nTotal readings across all shards: {total:,}")

## 6. Single-Shard Queries: Fast and Simple

When your query targets a single region, sharding shines. The query only hits one shard — less data, no contention with other regions.

In [ ]:
def explain_analyze(cursor, query, params=None):
    """Run EXPLAIN ANALYZE and print the query plan."""
    cursor.execute(f"EXPLAIN ANALYZE {query}", params)
    for row in cursor.fetchall():
        print(row[0])

In [ ]:
# Single-shard query: average temperature in East warehouse, March 2025
print("SINGLE-SHARD QUERY: East warehouse, March 2025 temperatures")
print("=" * 60)

east_cur = shard_curs['east']

east_cur.execute("""
    SELECT COUNT(*) AS readings,
           ROUND(AVG(value), 2) AS avg_temp,
           ROUND(MIN(value), 2) AS min_temp,
           ROUND(MAX(value), 2) AS max_temp
    FROM sensor_readings sr
    JOIN sensors s ON sr.sensor_id = s.sensor_id
    WHERE s.sensor_type = 'temperature'
      AND sr.reading_time >= '2025-03-01'
      AND sr.reading_time <  '2025-04-01'
""")

row = east_cur.fetchone()
print(f"Readings: {row[0]:,}")
print(f"Avg temp: {row[1]} F")
print(f"Min temp: {row[2]} F")
print(f"Max temp: {row[3]} F")

print("\nThis query only touched the East shard — 50K rows, not 150K.")

In [ ]:
# EXPLAIN ANALYZE on the single-shard query
print("EXPLAIN ANALYZE: Single-shard query (East)")
print("=" * 60)

explain_analyze(east_cur, """
    SELECT AVG(value)
    FROM sensor_readings
    WHERE reading_time >= '2025-03-01'
      AND reading_time <  '2025-04-01'
""")

print("\nFast! Only 50K rows on this shard, and the query is local.")

## 7. Cross-Shard JOINs: The Pain Begins

What if TechMart's CEO asks: *"Show me the hottest sensor across ALL warehouses this month."* This requires data from all three shards.

There is **no SQL JOIN that spans shards**. The databases are completely independent — they don't know each other exists.

In [ ]:
# Application-level cross-shard query
print("CROSS-SHARD QUERY: Hottest reading per region, March 2025")
print("=" * 60)

results = []

for region, cur in shard_curs.items():
    cur.execute("""
        SELECT sr.sensor_id, s.zone, sr.value, sr.reading_time
        FROM sensor_readings sr
        JOIN sensors s ON sr.sensor_id = s.sensor_id
        WHERE s.sensor_type = 'temperature'
          AND sr.reading_time >= '2025-03-01'
          AND sr.reading_time <  '2025-04-01'
        ORDER BY sr.value DESC
        LIMIT 1
    """)
    row = cur.fetchone()
    results.append((region, row[0], row[1], float(row[2]), row[3]))
    print(f"  {region:8s}: sensor {row[0]} in {row[1]:12s} -> {row[2]} F at {row[3]}")

# Find the global maximum in Python
hottest = max(results, key=lambda x: x[3])
print(f"\nGlobal hottest: {hottest[0]} region, sensor {hottest[1]} ({hottest[2]}), {hottest[3]} F")
print("\nThis required 3 separate queries + Python aggregation.")
print("A single-server query would be one SQL statement.")

## 8. Index Locality

Indexes are **shard-local**. An index on `reading_time` in the East shard knows nothing about readings in the West shard. Finding the global MAX requires querying every shard.

In [ ]:
# Create indexes on all shards
for region, cur in shard_curs.items():
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_readings_time
        ON sensor_readings (reading_time)
    """)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_readings_sensor_time
        ON sensor_readings (sensor_id, reading_time)
    """)

print("Created indexes on all 3 shards.")
print("That's 6 index operations (2 indexes × 3 shards).")
print("\nWith 50 shards, you'd need 100 index operations — and hope none fail.")

In [ ]:
# Finding the most recent reading — must query all shards
print("FINDING THE LATEST READING ACROSS ALL SHARDS")
print("=" * 60)

latest_per_shard = []
for region, cur in shard_curs.items():
    start = time.time()
    cur.execute("SELECT MAX(reading_time) FROM sensor_readings")
    max_time = cur.fetchone()[0]
    elapsed = time.time() - start
    latest_per_shard.append((region, max_time))
    print(f"  {region:8s}: {max_time}  ({elapsed*1000:.1f}ms)")

global_latest = max(latest_per_shard, key=lambda x: x[1])
print(f"\nGlobal latest: {global_latest[1]} (from {global_latest[0]} shard)")
print("\nEach shard's index makes the local MAX fast,")
print("but we still had to query all 3 shards to find the global MAX.")

## 9. Foreign Key Limitations

In a single database, foreign keys prevent orphan records — you can't reference a `sensor_id` that doesn't exist in the `sensors` table.

But with sharding, there are **no cross-shard foreign keys**. A shard can't verify that a sensor exists on another shard. Let's see what happens.

In [ ]:
# Insert an orphan reading — sensor_id 999 doesn't exist on ANY shard
print("FOREIGN KEY VIOLATION TEST")
print("=" * 60)

east_cur = shard_curs['east']

# This succeeds! No FK constraint to stop it.
east_cur.execute("""
    INSERT INTO sensor_readings (sensor_id, reading_time, value, unit)
    VALUES (999, '2025-03-15 12:00:00', 99.99, 'F')
""")

print("Inserted reading for sensor_id=999... which doesn't exist!")

# Verify the orphan exists
east_cur.execute("""
    SELECT sr.sensor_id, sr.value, sr.reading_time
    FROM sensor_readings sr
    LEFT JOIN sensors s ON sr.sensor_id = s.sensor_id
    WHERE s.sensor_id IS NULL
""")
orphans = east_cur.fetchall()
print(f"\nOrphan readings found: {len(orphans)}")
for o in orphans:
    print(f"  sensor_id={o[0]}, value={o[1]}, time={o[2]}")

print("\nIn a non-sharded database, the FK constraint would have blocked this INSERT.")
print("With sharding, data integrity becomes YOUR application's responsibility.")

In [ ]:
# Application-level FK enforcement
def insert_reading_safe(region, sensor_id, reading_time, value, unit):
    """Insert a reading with application-level FK validation."""
    cur = get_shard_cursor(region)
    
    # Check if sensor exists on this shard
    cur.execute("SELECT 1 FROM sensors WHERE sensor_id = %s", (sensor_id,))
    if cur.fetchone() is None:
        raise ValueError(
            f"FK violation: sensor_id={sensor_id} does not exist on shard '{region}'"
        )
    
    cur.execute(
        "INSERT INTO sensor_readings (sensor_id, reading_time, value, unit) "
        "VALUES (%s, %s, %s, %s)",
        (sensor_id, reading_time, value, unit)
    )
    return True

print("Application-level FK enforcement:")
print("=" * 60)

# This should succeed — sensor 101 exists on the east shard
try:
    insert_reading_safe('east', 101, '2025-03-15 13:00:00', 73.5, 'F')
    print("  sensor_id=101 on east: SUCCESS (sensor exists)")
except ValueError as e:
    print(f"  {e}")

# This should fail — sensor 999 doesn't exist
try:
    insert_reading_safe('east', 999, '2025-03-15 14:00:00', 88.0, 'F')
    print("  sensor_id=999 on east: SUCCESS")
except ValueError as e:
    print(f"  sensor_id=999 on east: BLOCKED — {e}")

# This should fail — sensor 201 exists but on central, not east
try:
    insert_reading_safe('east', 201, '2025-03-15 15:00:00', 71.0, 'F')
    print("  sensor_id=201 on east: SUCCESS")
except ValueError as e:
    print(f"  sensor_id=201 on east: BLOCKED — {e}")

print("\nYour application code must do what the database would normally handle.")

In [ ]:
# Clean up the orphan record
east_cur.execute("DELETE FROM sensor_readings WHERE sensor_id = 999")
print("Cleaned up orphan reading (sensor_id=999).")

### Cross-Shard FK Strategies: The Four Workarounds

Since foreign keys can't span shards, teams use different strategies depending on their data and query patterns. We've already seen two of these in action in this lab:

| Strategy | How It Works | We Saw It In... |
|----------|-------------|-----------------|
| **Reference Tables** | Copy small lookup tables to every shard | Section 3: `warehouses` replicated to all shards |
| **Co-location** | Shard all related tables by the same key | Our design: sensors + readings on the same shard by region |
| **Denormalization** | Copy referenced data into the row at insert time | Exercise 3 explores this approach |
| **Application Joins** | Query multiple shards, combine in app code | Sections 7 and 10: cross-shard queries in Python |

Our lab naturally uses **co-location** — sensors and their readings always land on the same shard (by region). That's why local JOINs between `sensors` and `sensor_readings` work on each shard. If we had sharded readings by `sensor_id` hash instead, a sensor's metadata and its readings could end up on different shards, breaking local JOINs.

## 10. Cross-Shard Aggregation

Business dashboards need aggregates across all regions. With sharding, this means querying every shard and combining results in your application.

In [ ]:
# Cross-shard aggregation: daily averages across all warehouses
print("CROSS-SHARD AGGREGATION: Temperature summary by region")
print("=" * 70)
print(f"{'Region':10s} {'Readings':>10s} {'Avg Temp':>10s} {'Min':>8s} {'Max':>8s} {'Alerts':>8s}")
print("-" * 70)

global_count = 0
global_sum = 0
global_min = float('inf')
global_max = float('-inf')
global_alerts = 0

for region, cur in shard_curs.items():
    cur.execute("""
        SELECT COUNT(*),
               ROUND(AVG(value), 2),
               ROUND(MIN(value), 2),
               ROUND(MAX(value), 2),
               COUNT(*) FILTER (WHERE alert_level = 'warning')
        FROM sensor_readings
        WHERE unit = 'F'
    """)
    row = cur.fetchone()
    cnt, avg_val, min_val, max_val, alerts = row
    
    print(f"{region:10s} {cnt:>10,} {avg_val:>10} {min_val:>8} {max_val:>8} {alerts:>8,}")
    
    global_count += cnt
    global_sum += float(avg_val) * cnt  # weighted sum for correct global avg
    global_min = min(global_min, float(min_val))
    global_max = max(global_max, float(max_val))
    global_alerts += alerts

print("-" * 70)
global_avg = round(global_sum / global_count, 2)
print(f"{'GLOBAL':10s} {global_count:>10,} {global_avg:>10} {global_min:>8} {global_max:>8} {global_alerts:>8,}")

print("\nNotice: computing the global AVG requires a WEIGHTED average,")
print("not just averaging the per-shard averages (that would be wrong")
print("if shards have different row counts).")

### Why Weighted Averages Matter

A common mistake when combining averages from different shards: simply averaging the per-shard averages. This gives the **wrong answer** when shards have different row counts. Let's prove it.

In [ ]:
# Demonstrate why averaging averages is WRONG
print("WEIGHTED vs NAIVE AVERAGE: Why it matters")
print("=" * 60)

# Collect per-shard stats
shard_stats = []
for region, cur in shard_curs.items():
    cur.execute("""
        SELECT COUNT(*), AVG(value)
        FROM sensor_readings WHERE unit = 'F'
    """)
    cnt, avg = cur.fetchone()
    shard_stats.append((region, cnt, float(avg)))
    print(f"  {region:8s}: {cnt:>6,} readings, avg = {avg}")

# WRONG: naive average of averages
naive_avg = sum(s[2] for s in shard_stats) / len(shard_stats)

# CORRECT: weighted average (sum of all values / total count)
weighted_avg = sum(s[1] * s[2] for s in shard_stats) / sum(s[1] for s in shard_stats)

print(f"\nNaive average  (avg of avgs):   {naive_avg:.4f}")
print(f"Weighted average (correct):     {weighted_avg:.4f}")
print(f"Difference:                     {abs(naive_avg - weighted_avg):.4f}")

print("\nIn this lab all shards have equal row counts, so the difference")
print("is small. But imagine one shard has 10x more data — the naive")
print("approach would give that shard's average equal weight to the others.")
print("\nRule: always use SUM(count * avg) / SUM(count), never AVG(avg).")

## 11. Coordinator with dblink: SQL-Level Cross-Shard Queries

Querying shards from Python works, but sometimes you want SQL-level access to all shards from a single connection point. PostgreSQL's `dblink` extension lets the **coordinator** query the shards directly.

This is not a real distributed query engine — it's essentially automating what we did in Python, but in SQL.

In [ ]:
# Set up dblink on the coordinator
coord_cur.execute("CREATE EXTENSION IF NOT EXISTS dblink")
print("dblink extension enabled on coordinator.")

In [ ]:
# Cross-shard query via dblink UNION ALL
print("COORDINATOR QUERY: All sensors across all shards (via dblink)")
print("=" * 60)

# Note: inside Docker, shards are accessed by container name, not localhost
coord_cur.execute("""
    SELECT * FROM dblink(
        'host=sharding_lab_shard1 dbname=shard1_db user=shard_user password=shard_pass',
        'SELECT sensor_id, warehouse_id, sensor_type, zone FROM sensors'
    ) AS t(sensor_id int, warehouse_id int, sensor_type varchar, zone varchar)
    UNION ALL
    SELECT * FROM dblink(
        'host=sharding_lab_shard2 dbname=shard2_db user=shard_user password=shard_pass',
        'SELECT sensor_id, warehouse_id, sensor_type, zone FROM sensors'
    ) AS t(sensor_id int, warehouse_id int, sensor_type varchar, zone varchar)
    UNION ALL
    SELECT * FROM dblink(
        'host=sharding_lab_shard3 dbname=shard3_db user=shard_user password=shard_pass',
        'SELECT sensor_id, warehouse_id, sensor_type, zone FROM sensors'
    ) AS t(sensor_id int, warehouse_id int, sensor_type varchar, zone varchar)
    ORDER BY sensor_id
""")

rows = coord_cur.fetchall()
print(f"{'ID':>4s}  {'WH':>3s}  {'Type':12s}  {'Zone'}")
print("-" * 40)
for r in rows:
    print(f"{r[0]:>4d}  {r[1]:>3d}  {r[2]:12s}  {r[3]}")

print(f"\nTotal: {len(rows)} sensors retrieved from 3 shards via coordinator")

In [ ]:
# Cross-shard aggregation via dblink
print("COORDINATOR QUERY: Reading count per shard (via dblink)")
print("=" * 60)

coord_cur.execute("""
    SELECT 'east' AS region, cnt FROM dblink(
        'host=sharding_lab_shard1 dbname=shard1_db user=shard_user password=shard_pass',
        'SELECT COUNT(*) FROM sensor_readings'
    ) AS t(cnt bigint)
    UNION ALL
    SELECT 'central', cnt FROM dblink(
        'host=sharding_lab_shard2 dbname=shard2_db user=shard_user password=shard_pass',
        'SELECT COUNT(*) FROM sensor_readings'
    ) AS t(cnt bigint)
    UNION ALL
    SELECT 'west', cnt FROM dblink(
        'host=sharding_lab_shard3 dbname=shard3_db user=shard_user password=shard_pass',
        'SELECT COUNT(*) FROM sensor_readings'
    ) AS t(cnt bigint)
""")

total = 0
for region, cnt in coord_cur.fetchall():
    print(f"  {region:8s}: {cnt:>8,} readings")
    total += cnt
print(f"  {'TOTAL':8s}: {total:>8,} readings")

print("\ndblink gives you SQL-level cross-shard access,")
print("but it's verbose and has no query optimization across shards.")
print("This is the complexity that distributed SQL databases automate.")

## 12. Summary: Partitioning vs Sharding

| | **Partitioning** (Lab 1) | **Sharding** (Lab 2) |
|---|---|---|
| **Where** | Single server | Multiple servers |
| **Managed by** | PostgreSQL engine | Your application |
| **JOINs** | Normal SQL | Application-level or dblink |
| **Foreign keys** | Fully enforced | Not possible across shards |
| **Indexes** | Per-partition, auto-managed | Per-shard, manually created |
| **Schema changes** | One ALTER TABLE | Must apply to every shard |
| **Aggregation** | Normal SQL | Query all shards + combine (weighted!) |
| **Data removal** | DROP PARTITION (instant) | DROP TABLE on one shard |
| **Scales** | Up to server limits | Horizontally (add more shards) |

### Key Takeaways

1. **Sharding trades simplicity for scalability** — you gain horizontal scale but lose SQL convenience
2. **Shard routing is application logic** — the database doesn't know about other shards
3. **Cross-shard queries are expensive** — both in latency and code complexity
4. **Data integrity is your responsibility** — no cross-shard FK constraints
5. **Operational overhead multiplies** — every schema change, index, and backup x N shards
6. **Schema drift is a real risk** — one missed ALTER TABLE and queries break on that shard
7. **Reference tables require manual sync** — stale copies lead to inconsistent query results
8. **Aggregations need care** — use weighted averages, not average-of-averages

### Cross-Shard FK Strategies Recap

| Strategy | Best For | Trade-off |
|----------|----------|-----------|
| **Reference Tables** | Small, rarely-changing lookup tables | Must sync updates to all shards |
| **Co-location** | Related data that's always queried together | Limits shard key flexibility |
| **Denormalization** | Data that rarely changes after insert | Storage duplication, stale data risk |
| **Application Joins** | Ad-hoc queries, analytics dashboards | Complex code, N+1 query risk |

## 13. Exercises

### Exercise 1: Hash-Based Routing
Modify the `hash_route` function to use SHA-256 instead of MD5. Route sensor IDs 1-20 and compare the distribution to MD5. Is the distribution more or less even?

### Exercise 2: Cross-Shard COUNT with Time Filter
Write a function that counts total readings across all shards for a given date range. Test it with March 2025. How does the count compare to querying a single shard?

### Exercise 3: Denormalization Strategy
Instead of a separate `sensors` table, what if we embedded `sensor_type` and `zone` directly into `sensor_readings`? Write the DDL for this denormalized table and discuss the trade-offs (storage vs query simplicity).

In [ ]:
# Space for your exercise work



## 14. Bridge to Week 05

Manual sharding works, but it's painful:

- You write and maintain all the routing logic
- Cross-shard queries require application code
- Schema changes must be applied to every shard
- Foreign keys and transactions don't span shards

**What if the database was designed for distribution from the ground up?**

Next week, we'll explore **Apache Cassandra** — a database that was built to distribute data automatically across nodes, with no single point of failure and no manual shard routing. But it makes very different trade-offs (CAP theorem) to achieve this.

## Cleanup

When you're done, shut down the Docker environment:

```bash
cd sharding-lab
docker compose down -v
```

In [ ]:
# Close all database connections
for region, conn in shard_conns.items():
    shard_curs[region].close()
    conn.close()
coord_cur.close()
coord_conn.close()

print("All connections closed.")